# File to run the carbonate chemistry routines (MOCSY) using thetao1, so1, talk1, dissic1 for piControl

author: Eike E. Köhn ......
date: Sep 5, 2025 ......
instructions: can be run with environment "conda env:miniforge3-py38" on spirit1

## Import packages

In [1]:
print('Loading packages...')
import sys
sys.path.append('../00_modules/.')
from import_packages import PackageGetter
globals().update(PackageGetter.import_standard_packages_for_analysis_and_plotting())
globals().update(PackageGetter.import_custom_packages(import_spline=False))

sys.path.append ('/home/ekoehn/software/co2sys/mocsy')
import mocsy

usage = """
    Usage: mocsy_ORCA2_da.py <DIC-file> <DIC-var> <Alk-file> <Alk-var> <Temp-file> <Temp-var> \\
                       <Salt-file> <Salt-var> <PO4-file> <PO4-var> <Si-file> <Si-var>                        
    Ouput file is created & stored following OCMIP5 (CMIP5) archive nomenclature 
    (under /prodigfsOCMIP5/derived_CMIP5). Its name is derived from DIC filename:
    * dir as for input dir but "dissic" replaed by output var names
    * filename handled likewise with "dissic" replaced by output var names"
""" 

Loading packages...


## Decide if figures are to be saved (and if so the directory)

In [2]:
savefigs = False
plot_dir = Plotter._get_plot_dir('standard')

## Set the parameters for the analysis of the preindustrial run

In [3]:
print('Setting parameters...')
run_params = Params.standard_set_piControl()  #Params.test_set_1pctCO2cdr() #

Setting parameters...


## Set the properties for the MOCSY calculations

In [4]:
print('Choosing variable and temporal resolution....')
temporal_resolution = 'annual_means'
depth_to_analyze = 1 # = 'surface'
use_woa2023_nutrients = 'all_models' # 'only_for_models_where_po4_and_si_are_not_available'

print('Specify MOCSY options (same for mocsy.mvars and mocsy.mderivnum) ....')
optcon='mol/m3' 
optt='Tpot' 
optp='m'
optb="u74"
optk1k2='w14' # 'l', 
optkf="dg"

calculate_vars = True
calculate_sensitivities = False

save_to_netcdf = True

Choosing variable and temporal resolution....
Specify MOCSY options (same for mocsy.mvars and mocsy.mderivnum) ....


# Loop over models to run mocsy


In [6]:
for key in run_params.keys():
    
    # get the parameters
    rparam = run_params[key]
    print(rparam.model,'...')

    # Dictionary to hold loaded data
    data = {}

    
    #################
    ## DRIVER VARs ##
    #################   
    # get the temperature, salinity, talk, dissic data (load explicitely into memory)
    for var in ['thetao','so','talk','dissic']:
        run_paths = ModelDataGetter._identify_path_strings(run_params,var,depth_to_analyze)
        path = run_paths[key]
        with xr.open_dataset(path) as ds:
            data[var] = ds[temporal_resolution].load()
    # get the shape of the thetao variable
    YEARS,LATS,LONS = data['thetao'].shape

    
    ###############
    ## NUTRIENTS ##
    ###############
    # get the nutrient data from woa2023 data by default and expand the data to have a year dimension
    with xr.open_dataset('/data/ekoehn/world_ocean_atlas/woa2023/phosphate/annual/woa23_all_p00_01_remapped.nc',decode_times=False) as woa_ds:
        data['po4'] = woa_ds['p_an'].isel(depth=0).squeeze().load() * 1025 / (10**6)
        data['po4'] = data['po4'].expand_dims({'year': YEARS}).transpose('year', 'lat', 'lon')
    with xr.open_dataset('/data/ekoehn/world_ocean_atlas/woa2023/silicate/annual/woa23_all_i00_01_remapped.nc',decode_times=False) as woa_ds:
        data['si'] = woa_ds['i_an'].isel(depth=0).squeeze().load() * 1025 / (10**6)
        data['si'] = data['si'].expand_dims({'year': YEARS}).transpose('year', 'lat', 'lon')
    # check if nutrient data is supposed to be overwritten by model nutrients
    if use_woa2023_nutrients != 'all_models' and rparam.model in ['NorESM2-LM','CNRM-ESM2-1']: # manually identified in advance
        for var in ['po4','si']:
            run_paths = ModelDataGetter._identify_path_strings(run_params,var,depth_to_analyze)
            path = run_paths[key]
            with xr.open_dataset(path) as ds:
                data[var] = ds[temporal_resolution].load()

    
    ##########################################
    ## ADD VARS (DEPTH, LAT, LON, PRESSURE) ## and expand them to have same dimensions as 3d variables
    ##########################################
    data['depth'] = data['thetao']['lev']
    if rparam.model == 'CESM2':
        data['depth'] = data['depth']/100 ## in CESM2 model depth is given in cm
    assert data['depth'] >= 0 
    assert data['depth'] < 10 
    data['lon'] = data['thetao']['lon']#.values
    data['lat'] = data['thetao']['lat']#.values
    data['lat'] = data['lat'].expand_dims({'year': YEARS, 'lon': LONS}).transpose('year', 'lat', 'lon')
    data['lon'] = data['lon'].expand_dims({'year': YEARS, 'lat': LATS}).transpose('year', 'lat', 'lon')
    data['depth'] = xr.DataArray(data['depth'], dims=()).expand_dims({'year': YEARS, 'lat': LATS, 'lon': LONS})
    data['patm'] = xr.DataArray(1, dims=()).expand_dims({'year': YEARS, 'lat': LATS, 'lon': LONS}) # set atm pressure to 1atm

    ##############################################################################################
    # RUN MOCSY to calculate variables, do some masking, put into xarray and add dimensions/coords
    ##############################################################################################
    if calculate_vars == True:
        print('.... calulate MOCSY vars.')
        #mocsy.mvars? - To show how the function works
        pH, pco2, fco2, co2, hco3, co3, OmegaA, OmegaC, BetaD, DENis, p, Tis = ( 
        mocsy.mvars(data['thetao'].values.flatten(), 
                    data['so'].values.flatten(), 
                    data['talk'].values.flatten(),
                    data['dissic'].values.flatten(), 
                    data['si'].values.flatten(), 
                    data['po4'].values.flatten(), 
                    data['patm'].values.flatten(), 
                    data['depth'].values.flatten(), 
                    data['lat'].values.flatten(), 
                    optcon=optcon, 
                    optt=optt, 
                    optp=optp, 
                    optb=optb, 
                    optk1k2=optk1k2, #'l', 
                    optkf=optkf)
        )
    
        mocsy_output = {}
        mocsy_output['ph'] = np.reshape(pH,(YEARS,LATS,LONS))
        mocsy_output['pco2'] = np.reshape(pco2,(YEARS,LATS,LONS))
        mocsy_output['co3'] = np.reshape(co3,(YEARS,LATS,LONS))
        mocsy_output['omegaa'] = np.reshape(OmegaA,(YEARS,LATS,LONS))
        mocsy_output['denis'] = np.reshape(DENis,(YEARS,LATS,LONS))
        #mocsy_output['fco2'] = np.reshape(fco2,(YEARS,LATS,LONS))
        #mocsy_output['co2'] = np.reshape(co2,(YEARS,LATS,LONS))
        #mocsy_output['hco3'] = np.reshape(hco3,(YEARS,LATS,LONS))
        #mocsy_output['omegac'] = np.reshape(OmegaC,(YEARS,LATS,LONS))
        #mocsy_output['betad'] = np.reshape(BetaD,(YEARS,LATS,LONS))
        #mocsy_output['p'] =  np.reshape(p,(YEARS,LATS,LONS))
        #mocsy_output['tis'] = np.reshape(Tis,(YEARS,LATS,LONS))
    
        for var in mocsy_output.keys():
            masker = mocsy_output[var]>1e19
            mocsy_output[var][masker] = np.NaN
            mocsy_output[var] = xr.DataArray(mocsy_output[var],dims=data['thetao'].dims,coords=data['thetao'].coords)
                    
        ## Save the MOCSY output for in situ density, pH, OmegaA, pco2 and co3
        if save_to_netcdf == True:   
            for mocsy_var in mocsy_output.keys():
                # put into a dataset of its own
                ds_mocsy_var = mocsy_output[mocsy_var].to_dataset(name='annual_means')
                # add attributes
                ds_mocsy_var.attrs['temporal_resolution'] = temporal_resolution
                ds_mocsy_var.attrs['depth_to_analyze'] = depth_to_analyze
                ds_mocsy_var.attrs['use_woa2023_nutrients'] = use_woa2023_nutrients
                ds_mocsy_var.attrs['optcon'] = optcon
                ds_mocsy_var.attrs['optt'] = optt
                ds_mocsy_var.attrs['optp'] = optp
                ds_mocsy_var.attrs['optb'] = optb
                ds_mocsy_var.attrs['optk1k2'] = optk1k2
                ds_mocsy_var.attrs['optkf'] = optkf
                ds_mocsy_var.attrs['author'] = 'Eike E. Köhn'
                ds_mocsy_var.attrs['date'] = datetime.date.today().isoformat()
                
                # now save this dataset
                save_dir = f'/data/ekoehn/projects/arctic_acidification_reversibility/data/processed_data_piControl/mocsy_output/{mocsy_var}{depth_to_analyze}'
                os.makedirs(save_dir, exist_ok=True)
                save_filename = f'{mocsy_var}{depth_to_analyze}_{rparam.model}_processed.nc'
                ds_mocsy_var.to_netcdf(f'{save_dir}/{save_filename}')


    ##################################################################################################
    # RUN MOCSY to calculate sensitivities, do some masking, put into xarray and add dimensions/coords
    ##################################################################################################
    if calculate_sensitivities == True:
        print('.... calulate MOCSY sensitivities.')
    
        mocsy_sensitivities = {}
        for invar in ['alk','dic','tem','sal']: #,'pho','sil','k0 ','k1 ','k2 ','kb ','kw ','ka ','kc ']:
            print(f'........ sensitivity wrt {invar}')
            #mocsy.mderivnum? - To show how the function works
            dh_dx,dpco2_dx,dfco2_dx,dco2_dx,dhco3_dx,dco3_dx,domegaa_dx,domegac_dx = ( 
            mocsy.mderivnum(data['thetao'].values.flatten(), 
                        data['so'].values.flatten(), 
                        data['talk'].values.flatten(),
                        data['dissic'].values.flatten(), 
                        data['si'].values.flatten(), 
                        data['po4'].values.flatten(), 
                        data['patm'].values.flatten(), 
                        data['depth'].values.flatten(), 
                        data['lat'].values.flatten(), 
                        invar, # 
                        optcon=optcon, 
                        optt=optt, 
                        optp=optp, 
                        optb=optb, 
                        optk1k2=optk1k2, # 'l', 
                        optkf=optkf)
            )
            mocsy_sensitivities[f'dh_d{invar}'] = np.reshape(dh_dx,(YEARS,LATS,LONS))
            mocsy_sensitivities[f'dpco2_d{invar}'] = np.reshape(dpco2_dx,(YEARS,LATS,LONS))
            mocsy_sensitivities[f'dco3_d{invar}'] = np.reshape(dco3_dx,(YEARS,LATS,LONS))
            mocsy_sensitivities[f'domegaa_d{invar}'] = np.reshape(domegaa_dx,(YEARS,LATS,LONS))
            #mocsy_sensitivities[f'dfco2_d{invar}'] = np.reshape(dfco2_dx,(YEARS,LATS,LONS))
            #mocsy_sensitivities[f'dco2_d{invar}'] = np.reshape(dco2_dx,(YEARS,LATS,LONS))
            #mocsy_sensitivities[f'dhco3_d{invar}'] = np.reshape(dhco3_dx,(YEARS,LATS,LONS))
            #mocsy_sensitivities[f'domegac_d{invar}'] = np.reshape(domegac_dx,(YEARS,LATS,LONS))
    
        for var in mocsy_sensitivities.keys():
            masker = (mocsy_sensitivities[var]>1e19)+(mocsy_sensitivities[var]<-2e25)
            mocsy_sensitivities[var][masker] = np.NaN
            mocsy_sensitivities[var] = xr.DataArray(mocsy_sensitivities[var],dims=data['thetao'].dims,coords=data['thetao'].coords)
    
    
        ## Save the MOCSY sensitivities for in situ density, pH, OmegaA, pco2 and co3
        if save_to_netcdf == True:   
            for mocsy_sens in mocsy_sensitivities.keys():
                # put into a dataset of its own
                ds_mocsy_sensitivity = mocsy_sensitivities[mocsy_sens].to_dataset(name='annual_means')
                # add attributes
                ds_mocsy_sensitivity.attrs['temporal_resolution'] = temporal_resolution
                ds_mocsy_sensitivity.attrs['depth_to_analyze'] = depth_to_analyze
                ds_mocsy_sensitivity.attrs['use_woa2023_nutrients'] = use_woa2023_nutrients
                ds_mocsy_sensitivity.attrs['optcon'] = optcon
                ds_mocsy_sensitivity.attrs['optt'] = optt
                ds_mocsy_sensitivity.attrs['optp'] = optp
                ds_mocsy_sensitivity.attrs['optb'] = optb
                ds_mocsy_sensitivity.attrs['optk1k2'] = optk1k2
                ds_mocsy_sensitivity.attrs['optkf'] = optkf
                ds_mocsy_sensitivity.attrs['author'] = 'Eike E. Köhn'
                ds_mocsy_sensitivity.attrs['date'] = datetime.date.today().isoformat()
                # now save this dataset
                save_dir = f'/data/ekoehn/projects/arctic_acidification_reversibility/data/processed_data_piControl/mocsy_sensitivities/{mocsy_sens}{depth_to_analyze}'
                os.makedirs(save_dir, exist_ok=True)
                save_filename = f'{mocsy_sens}{depth_to_analyze}_{rparam.model}_processed.nc'
                ds_mocsy_sensitivity.to_netcdf(f'{save_dir}/{save_filename}')


UKESM1-0-LL ...
.... calulate MOCSY vars.
NorESM2-LM ...
.... calulate MOCSY vars.
MIROC-ES2L ...
.... calulate MOCSY vars.
CNRM-ESM2-1 ...
.... calulate MOCSY vars.
ACCESS-ESM1-5 ...
.... calulate MOCSY vars.
CESM2 ...
.... calulate MOCSY vars.
CanESM5 ...
.... calulate MOCSY vars.
GFDL-ESM4 ...
.... calulate MOCSY vars.
